# 01 — GLM-OCR Base Benchmark (Google Colab Pro)

Establishes the **pre-fine-tuning** baseline for `zai-org/GLM-OCR` on the frozen UIT-HWDB-line split.

Official basis:
- Transformers GLM-OCR documentation: https://huggingface.co/docs/transformers/main/model_doc/glm_ocr
- Official GLM-OCR model: https://huggingface.co/zai-org/GLM-OCR
- Official text-recognition prompt: `Text Recognition:`

Protocol: fixed 20-sample validation smoke → full 682 validation → freeze inference settings → one-time 201 test baseline. The test set must not be used to tune the prompt or decoding.

## 0. Install dependencies

In [ ]:
%pip install -q -U "kagglehub>=1.0.2" "jiwer>=4.0.0" pandas pillow
%pip install -q -U "transformers>=5.3.0,<5.18" "accelerate>=1.2.0" huggingface_hub

## 1. Mount Drive and load the frozen data

In [ ]:
import os, json, time, random, platform, unicodedata, gc, math, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from jiwer import cer, wer

from google.colab import drive
drive.mount('/content/drive')

# Optional Colab secret. KaggleHub can also prompt/authenticate through its normal flow.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

import kagglehub

SEED = 42
RAW_HANDLE = 'ntklinhfitus/uit-hwdb'
MANIFEST_HANDLE = 'ntklinhfitus/uit-hwdb-manifest'
PROJECT_ROOT = Path('/content/drive/MyDrive/vlm_handwriting_ocr')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Try a partial raw download first. Fall back to the full Kaggle dataset if the
# installed KaggleHub/runtime does not accept directory-level download.
try:
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE, path='UIT_HWDB_line'))
except Exception as e:
    print('Partial raw download unavailable, falling back to full dataset:', repr(e))
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE))
manifest_download = Path(kagglehub.dataset_download(MANIFEST_HANDLE))

def locate_raw_line_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    candidates=[p for p in pool if (p/'train_data').is_dir() and (p/'test_data').is_dir()]
    assert candidates, f'Cannot locate UIT-HWDB-line train_data/test_data under {base}'
    candidates.sort(key=lambda p: ('UIT_HWDB_line' not in str(p), len(str(p))))
    return candidates[0]

def locate_manifest_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    for p in pool:
        if all((p/f).exists() for f in ['train.csv','val.csv','test.csv']):
            return p
    raise FileNotFoundError(f'Cannot locate train.csv/val.csv/test.csv under {base}')

RAW_ROOT=locate_raw_line_root(raw_download)
MANIFEST_ROOT=locate_manifest_root(manifest_download)
print('RAW_ROOT      =',RAW_ROOT)
print('MANIFEST_ROOT =',MANIFEST_ROOT)

In [ ]:
train_df=pd.read_csv(MANIFEST_ROOT/'train.csv')
val_df=pd.read_csv(MANIFEST_ROOT/'val.csv')
test_df=pd.read_csv(MANIFEST_ROOT/'test.csv')
EXPECTED={'train':6346,'validation':682,'test':201}
assert len(train_df)==EXPECTED['train'],len(train_df)
assert len(val_df)==EXPECTED['validation'],len(val_df)
assert len(test_df)==EXPECTED['test'],len(test_df)
required={'writer_id','filename','relative_path','text'}
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=required-set(frame.columns)
    assert not missing,f'{name} missing columns: {missing}'
train_writers=set(train_df.writer_id); val_writers=set(val_df.writer_id); test_writers=set(test_df.writer_id)
assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

def resolve_image_path(row):
    return RAW_ROOT/str(row['relative_path'])
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=[str(resolve_image_path(r)) for _,r in frame.iterrows() if not resolve_image_path(r).exists()]
    assert not missing,f'{name}: missing image paths, e.g. {missing[:3]}'
print(f'Train      : {len(train_df)} samples | {len(train_writers)} writers')
print(f'Validation : {len(val_df)} samples | {len(val_writers)} writers')
print(f'Test       : {len(test_df)} samples | {len(test_writers)} writers')
print('✅ Frozen writer-disjoint split verified.')

In [ ]:
def normalize_for_eval(text):
    # Strict OCR evaluation: Unicode NFC only.
    return unicodedata.normalize('NFC',str(text))

def compute_metrics(gt_list,pred_list):
    if len(gt_list)!=len(pred_list) or len(gt_list)==0:
        raise ValueError('GT/prediction lists must have the same non-zero length.')
    gt=[normalize_for_eval(x) for x in gt_list]
    pred=[normalize_for_eval(x) for x in pred_list]
    exact=sum(g==p for g,p in zip(gt,pred))/len(gt)
    return {'CER':float(cer(gt,pred)),'WER':float(wer(gt,pred)),'Exact_Line_Accuracy':float(exact),'N':int(len(gt))}
assert compute_metrics(['Việt Nam'],['Việt Nam'])['CER']==0.0
assert compute_metrics(['Biển Đông.'],['Biển đông.'])['CER']>0.0
SMOKE_SIZE=20
smoke_df=val_df.sample(n=SMOKE_SIZE,random_state=SEED).sort_index().reset_index(drop=True)
print('✅ Strict evaluator + fixed 20-sample validation smoke set ready.')

## 2. Optional image/label sanity check

In [ ]:
import matplotlib.pyplot as plt
sample=pd.concat([
    train_df.sample(1,random_state=SEED).assign(_split='train'),
    val_df.sample(2,random_state=SEED).assign(_split='validation'),
    test_df.sample(1,random_state=SEED).assign(_split='test'),
])
for _,row in sample.iterrows():
    img=Image.open(resolve_image_path(row)).convert('RGB')
    plt.figure(figsize=(14,2.5)); plt.imshow(img); plt.axis('off')
    plt.title(f"{row['_split']} | writer={row['writer_id']} | {row['filename']}\nGT: {row['text']}")
    plt.show()

## 3. GPU and benchmark configuration

In [ ]:
import torch
assert torch.cuda.is_available(),'Colab GPU required.'
torch.manual_seed(SEED)
MODEL_ID='zai-org/GLM-OCR'; MODEL_NAME='GLM-OCR'; PROMPT='Text Recognition:'; MAX_NEW_TOKENS=256
RUN_FULL_VALIDATION=False
RUN_TEST_BASELINE=False
RESULT_DIR=PROJECT_ROOT/'results'/'glm_ocr'/'base'; RESULT_DIR.mkdir(parents=True,exist_ok=True)
smoke_df.to_csv(RESULT_DIR/'smoke_val_20.csv',index=False)
GPU_NAME=torch.cuda.get_device_name(0); BF16=torch.cuda.is_bf16_supported(); DTYPE=torch.bfloat16 if BF16 else torch.float16
print('GPU:',GPU_NAME,'| BF16:',BF16,'| dtype:',DTYPE)

## 4. Load the official base model

In [ ]:
from transformers import AutoProcessor, GlmOcrForConditionalGeneration
processor=AutoProcessor.from_pretrained(MODEL_ID)
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
model=GlmOcrForConditionalGeneration.from_pretrained(MODEL_ID,dtype=DTYPE,device_map='auto').eval()
MODEL_DEVICE=next(model.parameters()).device
print('device=',MODEL_DEVICE)
print(f'allocated after load={torch.cuda.memory_allocated()/1024**3:.2f} GB')

## 5. Inference helper

In [ ]:
@torch.inference_mode()
def predict_one(image_path):
    messages=[{'role':'user','content':[{'type':'image','path':str(image_path)},{'type':'text','text':PROMPT}]}]
    inputs=processor.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors='pt')
    inputs.pop('token_type_ids',None); inputs=inputs.to(MODEL_DEVICE)
    torch.cuda.synchronize(); t0=time.perf_counter()
    output=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.1)
    torch.cuda.synchronize(); latency=time.perf_counter()-t0
    n=inputs['input_ids'].shape[-1]
    pred=processor.decode(output[0][n:],skip_special_tokens=True).strip()
    return pred,latency

def run_benchmark(frame,split_name):
    rows=[]; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); started=time.perf_counter()
    for i,(_,row) in enumerate(frame.iterrows(),1):
        pred,lat=predict_one(resolve_image_path(row)); gt=normalize_for_eval(row['text']); pred=normalize_for_eval(pred)
        rows.append({'model':MODEL_NAME,'split':split_name,'writer_id':int(row['writer_id']),'filename':row['filename'],'relative_path':row['relative_path'],'ground_truth':gt,'prediction':pred,'sample_CER':float(cer(gt,pred)),'sample_WER':float(wer(gt,pred)),'exact_match':bool(gt==pred),'latency_sec':float(lat)})
        if i<=20 or i%50==0 or i==len(frame): print(f'[{i}/{len(frame)}] CER={rows[-1]["sample_CER"]:.3f} | {lat:.3f}s')
    out=pd.DataFrame(rows); metrics=compute_metrics(out.ground_truth.tolist(),out.prediction.tolist())
    metrics.update({'model':MODEL_NAME,'split':split_name,'total_runtime_sec':float(time.perf_counter()-started),'latency_mean_sec':float(out.latency_sec.mean()),'latency_p50_sec':float(out.latency_sec.quantile(.5)),'latency_p90_sec':float(out.latency_sec.quantile(.9)),'peak_gpu_vram_gb':float(torch.cuda.max_memory_allocated()/1024**3),'gpu_name':GPU_NAME,'dtype':str(DTYPE),'prompt':PROMPT,'max_new_tokens':MAX_NEW_TOKENS,'seed':SEED})
    return out,metrics

## 6. Single-sample smoke

In [ ]:
row=smoke_df.iloc[0]; pred,lat=predict_one(resolve_image_path(row))
print('GT  :',row['text']); print('PRED:',pred); print('latency=',round(lat,3),'s'); print('CER=',cer(normalize_for_eval(row['text']),normalize_for_eval(pred)))

## 7. Fixed 20-sample validation smoke benchmark

In [ ]:
smoke_predictions,smoke_metrics=run_benchmark(smoke_df,'validation_smoke20')
display(smoke_predictions[['writer_id','filename','ground_truth','prediction','sample_CER','latency_sec']])
print(json.dumps(smoke_metrics,ensure_ascii=False,indent=2))
smoke_predictions.to_csv(RESULT_DIR/'glm_ocr_base_smoke20_predictions.csv',index=False)
(RESULT_DIR/'glm_ocr_base_smoke20_metrics.json').write_text(json.dumps(smoke_metrics,ensure_ascii=False,indent=2),encoding='utf-8')

## 8. Inspection gate
Inspect the 20 predictions. Confirm that output is OCR text only, Vietnamese Unicode is intact, and there is no repetitive collapse. Only then enable full validation below.

## 9. Full validation — 682 samples

In [ ]:
RUN_FULL_VALIDATION=False  # manually set True after smoke inspection
if RUN_FULL_VALIDATION:
    val_predictions,val_metrics=run_benchmark(val_df.reset_index(drop=True),'validation')
    val_predictions.to_csv(RESULT_DIR/'glm_ocr_base_val_predictions.csv',index=False)
    (RESULT_DIR/'glm_ocr_base_val_metrics.json').write_text(json.dumps(val_metrics,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(val_metrics,ensure_ascii=False,indent=2))
else: print('Full validation gated.')

## 10. Frozen test baseline — 201 samples

In [ ]:
RUN_TEST_BASELINE=False  # set True only after validation config is frozen
if RUN_TEST_BASELINE:
    test_predictions,test_metrics=run_benchmark(test_df.reset_index(drop=True),'test')
    test_predictions.to_csv(RESULT_DIR/'glm_ocr_base_test_predictions.csv',index=False)
    (RESULT_DIR/'glm_ocr_base_test_metrics.json').write_text(json.dumps(test_metrics,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(test_metrics,ensure_ascii=False,indent=2))
else: print('Frozen test remains untouched.')

## 11. Save provenance

In [ ]:
import transformers
prov={'model_id':MODEL_ID,'prompt':PROMPT,'seed':SEED,'split_counts':EXPECTED,'gpu':GPU_NAME,'dtype':str(DTYPE),'python':platform.python_version(),'torch':torch.__version__,'transformers':transformers.__version__}
(RESULT_DIR/'environment.json').write_text(json.dumps(prov,indent=2),encoding='utf-8'); print('Saved to',RESULT_DIR)